
# Step 4 - subset spawning location dataframes etc:
Input: data from step 3 

Filter for when there are 21 days spent in a location as “used”
export
Output:  'IDLloc_spawning_21dfilter.csv'

summary.to_csv(output_path + 'nursery_use_summary.csv', index=False)


In [3]:
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta
import os

from shapely.geometry import Point, Polygon as ShapelyPolygon
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# File paths to data
repo_path = '/Users/zephyrsylvester/repos/circumpolar-connectivity-analysis/data/'
output_path = '/Users/zephyrsylvester/repos/circumpolar-connectivity-analysis/processed_data/'
fig_path = '/Users/zephyrsylvester/repos/circumpolar-connectivity-analysis/figures/'

# List of simulations and locations
sim_list = ['007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018']
locations = ['BS', 'GERL', 'GP', 'MB2']


In [5]:
# file = output_path + 'processed_trajectories_sub3filt.csv'
file = output_path + 'processed_trajectories_21dfilter.csv'
filtered_df=pd.read_csv(file) 
print('raw number of larvae:', filtered_df.larval_id.nunique())
filtered_df

raw number of larvae: 5002


,N,T,date,lat,lon,release,stage,larval_id,start_year,hypothesis,IDL_loc,BS,GERL,GP,MB2,outside
0,0,0,2016-11-01,-66.912544,287.83374,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
1,0,1,2016-11-02,-66.830450,288.03165,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
2,0,2,2016-11-03,-66.757600,288.15454,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
3,0,3,2016-11-04,-66.711136,288.16013,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
4,0,4,2016-11-05,-66.718590,288.06876,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900355,199,175,2019-09-12,-60.327457,306.02032,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
900356,199,176,2019-09-13,-60.238560,305.90730,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
900357,199,177,2019-09-14,-60.116886,305.83090,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
900358,199,178,2019-09-15,-60.023293,305.82330,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True


In [6]:
# Create a summary of the number of unique larval_id's for each hypothesis, start year, and IDL_loc
summary_df = filtered_df.groupby(['hypothesis', 'start_year', 'IDL_loc'])['larval_id'].nunique().reset_index()

# Rename the column for clarity
summary_df.rename(columns={'larval_id': 'unique_larval_count'}, inplace=True)

summary_df.head()

,hypothesis,start_year,IDL_loc,unique_larval_count
0,h_dvm,2016,BS,71
1,h_dvm,2016,GERL,58
2,h_dvm,2016,GP,73
3,h_dvm,2016,MB2,30
4,h_dvm,2017,BS,78


# Extract Starting Locations

In [7]:
# Extract Summary Information
critical_columns = ['larval_id', 'date', 'lon', 'lat', 'hypothesis', 'start_year', 'IDL_loc', 'stage']
presence_columns = ['BS', 'GERL', 'GP', 'MB2','outside']

# Function to create the starting location DataFrame
def create_starting_location_df(df):
    t0_data = df[df['T'] == 0].copy()
    starting_locations = t0_data[critical_columns+presence_columns].copy()  # Use copy() to avoid SettingWithCopyWarning
    starting_locations.rename(columns={'date': 'release_date', 'lat': 'release_lat', 'lon': 'release_lon'}, inplace=True)
    return starting_locations

In [8]:
# Create the starting location DataFrame
starting_location_df = create_starting_location_df(filtered_df)

# Check It
print('number of larvae:', starting_location_df.larval_id.nunique())
# presence_summary_starting_location = summarize_presence(starting_location_df)
starting_location_df.head()

number of larvae: 5002


,larval_id,release_date,release_lon,release_lat,hypothesis,start_year,IDL_loc,stage,BS,GERL,GP,MB2,outside
0,00_16_1_0001,2016-11-01,287.83374,-66.912544,h_null,2016,BS,0,False,False,False,False,True
180,00_16_1_0002,2016-11-01,292.12890,-64.621430,h_null,2016,BS,0,False,False,False,False,True
360,00_16_1_0017,2016-11-01,302.04650,-62.609480,h_null,2016,BS,0,False,False,False,False,True
540,00_16_1_0018,2016-11-01,302.19672,-62.309680,h_null,2016,BS,0,False,False,False,False,True
720,00_16_1_0022,2016-11-15,289.22250,-65.792010,h_null,2016,BS,0,False,False,False,False,True


In [9]:
# Create a summary of the number of unique larval_id's for each hypothesis, start year, and IDL_loc
summary_sg = starting_location_df.groupby(['hypothesis', 'start_year', 'IDL_loc'])['larval_id'].nunique().reset_index()

# Rename the column for clarity
summary_sg.rename(columns={'larval_id': 'unique_larval_count'}, inplace=True)

summary_sg.head()

,hypothesis,start_year,IDL_loc,unique_larval_count
0,h_dvm,2016,BS,71
1,h_dvm,2016,GERL,58
2,h_dvm,2016,GP,73
3,h_dvm,2016,MB2,30
4,h_dvm,2017,BS,78


In [10]:
def compare_dataframes(df1, df2):
    if df1.equals(df2):
        print("The DataFrames are exactly the same.")
    else:
        print("The DataFrames are not the same.")

# Example usage:
compare_dataframes(summary_sg, summary_df)

The DataFrames are exactly the same.


In [10]:
# Save the starting locations DataFrame to a CSV file
starting_location_df.to_csv(output_path + 'IDLloc_spawning_21dfilter.csv', index=False)

# Save the location summary DataFrame to a CSV file
summary_sg.to_csv(output_path + 'nursery_use_summary.csv', index=False)

In [11]:
# # Save the starting locations DataFrame to a CSV file
# starting_location_df.to_csv(output_path + 'IDLloc_spawning_sub3filt.csv', index=False)

# # Save the location summary DataFrame to a CSV file
# summary_sg.to_csv(output_path + 'nursery_use_summary_sub3filt.csv', index=False)